# TFT (with exogenous features)

The Temporal Fusion Transformer can use exogenous features (unlike DLinear, N-BEATS
and PatchTST, which are univariate here). We test whether adding features helps by
training TFT twice: once on past sales only, once with historical + static features.

WMAE is measured on the real validation rows, comparable to the tree models. Logged to
the shared DagsHub.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import yaml
import warnings
import numpy as np
import pandas as pd
import mlflow
import dagshub

from src.features.nn_preprocessing import (
    build_long_df, temporal_split, get_real_validation, attach_static, FREQ, HIST_EXOG,
)
from src.models.tft_pipeline import build_tft
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings("ignore")
STAT_EXOG = ["Type_code", "Size"]

# Shared DagsHub tracking.
dagshub.init(repo_owner="slosa23", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("TFT_Training")

# Load the model config.
with open("configs/tft_config.yaml") as f:
    config = yaml.safe_load(f)
params = config["model"]["params"]
split_date = config["data"]["split_date"]
print("Tracking URI:", mlflow.get_tracking_uri())

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-09 21:55:02,002	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-07-09 21:55:02,483	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Accessing as GigiSichinava

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

2026/07/09 21:55:06 INFO mlflow.tracking.fluent: Experiment with name 'TFT_Training' does not exist. Creating a new experiment.


Tracking URI: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow


In [3]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
print("train:", train_raw.shape)

train: (421570, 5)


## Run 1: Data Preparation

In [4]:
with mlflow.start_run(run_name="TFT_Data_Prep"):
    Y_df = build_long_df(train_raw, stores, features, fill_gaps=True)
    static_df = attach_static(Y_df, stores)
    train_df, valid_df, horizon = temporal_split(Y_df, split_date)
    real_valid = get_real_validation(train_raw, stores, features, split_date)

    mlflow.log_param("freq", FREQ)
    mlflow.log_param("split_date", split_date)
    mlflow.log_param("n_series", int(Y_df["unique_id"].nunique()))
    mlflow.log_param("horizon_weeks", int(horizon))
    mlflow.log_param("real_valid_rows", int(len(real_valid)))

    print("series:", Y_df["unique_id"].nunique(), "| horizon:", horizon,
          "| real valid rows:", len(real_valid))

series: 3331 | horizon: 43 | real valid rows: 127438
🏃 View run TFT_Data_Prep at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/7/runs/b0a461631a6d4d8c8d43f33afc934b86
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/7


## WMAE on real validation rows

In [5]:
def evaluate_on_real(forecast_df, real_valid, model_col="TFT"):
    merged = real_valid.merge(
        forecast_df[["unique_id", "ds", model_col]], on=["unique_id", "ds"], how="inner",
    )
    preds = merged[model_col].clip(lower=0)  # sales can't be negative
    wmae = calculate_wmae(merged["y"], preds, merged["IsHoliday"])
    return wmae, merged

## Run 2: TFT univariate (past sales only)

In [6]:
with mlflow.start_run(run_name="TFT_Univariate"):
    mlflow.log_params(params)
    mlflow.log_param("exogenous", "none")

    nf = build_tft(horizon, params, freq=FREQ)
    nf.fit(df=train_df[["unique_id", "ds", "y"]])

    fcst = nf.predict()
    wmae, _ = evaluate_on_real(fcst, real_valid, "TFT")
    mlflow.log_metric("WMAE", float(wmae))
    print(f"TFT univariate WMAE (real validation): {wmae:,.2f}")

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | padder_train            | ConstantPad1d            | 0      | train
2 | scaler                  | TemporalNorm             | 0      | train
3 | embedding               | TFTEmbedding             | 128    | train
4 | temporal_encoder        | TemporalCovariateEncoder | 39.6 K | train
5 | temporal_fusion_decoder | TemporalFusionDecoder    | 17.0 K | train
6 | output_adapter          | Linear                   | 33     | train
-----------------------------------------------------------------------------
56.8 K    Trainable params
0         Non-trainable params
56.8 K    Total params
0.227     Total estimated model params size (MB)
88        Modules in train mode
0         Modul

Epoch 0:  96%|█████████▌| 100/104 [02:12<00:05,  0.75it/s, v_num=6, train_loss_step=73.10]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 96/96 [02:13<00:00,  0.72it/s, v_num=6, train_loss_step=53.50, train_loss_epoch=114.0]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 96/96 [02:13<00:00,  0.72it/s, v_num=6, train_loss_step=53.50, train_loss_epoch=93.00]

`Trainer.fit` stopped: `max_steps=200` reached.


Epoch 1: 100%|██████████| 96/96 [02:13<00:00,  0.72it/s, v_num=6, train_loss_step=53.50, train_loss_epoch=93.00]


GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Predicting DataLoader 0: 100%|██████████| 104/104 [00:03<00:00, 27.71it/s]
TFT univariate WMAE (real validation): 2,360.51
🏃 View run TFT_Univariate at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/7/runs/ee156354ce694facb6c33a4f79e20199
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/7


## Run 3: TFT with exogenous features

Adds historical (temperature, fuel, CPI, unemployment, markdowns) and static (store
type, size) features. Earlier this did not beat the univariate version.

In [7]:
with mlflow.start_run(run_name="TFT_With_Exogenous"):
    mlflow.log_params(params)
    mlflow.log_param("exogenous", "hist+static")

    nf = build_tft(horizon, params, hist_exog=HIST_EXOG, stat_exog=STAT_EXOG, freq=FREQ)
    nf.fit(df=train_df[["unique_id", "ds", "y"] + HIST_EXOG], static_df=static_df)

    fcst = nf.predict()
    wmae, _ = evaluate_on_real(fcst, real_valid, "TFT")
    mlflow.log_metric("WMAE", float(wmae))
    print(f"TFT with-exogenous WMAE (real validation): {wmae:,.2f}")

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | padder_train            | ConstantPad1d            | 0      | train
2 | scaler                  | TemporalNorm             | 0      | train
3 | embedding               | TFTEmbedding             | 896    | train
4 | static_encoder          | StaticCovariateEncoder   | 30.2 K | train
5 | temporal_encoder        | TemporalCovariateEncoder | 97.9 K | train
6 | temporal_fusion_decoder | TemporalFusionDecoder    | 17.0 K | train
7 | output_adapter          | Linear                   | 33     | train
-----------------------------------------------------------------------------
145 K     Trainable params
0         Non-trainable params
145 K     Total params
0.584     Total estimate

Epoch 0:  96%|█████████▌| 100/104 [05:11<00:12,  0.32it/s, v_num=8, train_loss_step=16.50] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 96/96 [05:23<00:00,  0.30it/s, v_num=8, train_loss_step=17.30, train_loss_epoch=123.0]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 96/96 [05:24<00:00,  0.30it/s, v_num=8, train_loss_step=17.30, train_loss_epoch=103.0]

`Trainer.fit` stopped: `max_steps=200` reached.


Epoch 1: 100%|██████████| 96/96 [05:24<00:00,  0.30it/s, v_num=8, train_loss_step=17.30, train_loss_epoch=103.0]


GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Predicting DataLoader 0: 100%|██████████| 104/104 [00:05<00:00, 17.75it/s]
TFT with-exogenous WMAE (real validation): 2,391.82
🏃 View run TFT_With_Exogenous at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/7/runs/ceddeffbd8bd44ceb0c62c5657f084c7
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/7


## Run 4: Best pipeline (retrain on all data, save)

Univariate was better, so the saved model is univariate.

In [8]:
with mlflow.start_run(run_name="TFT_Best_Pipeline"):
    mlflow.log_params(params)
    mlflow.log_param("exogenous", "none")

    nf_best = build_tft(horizon, params, freq=FREQ)
    nf_best.fit(df=Y_df[["unique_id", "ds", "y"]])  # full history

    save_path = "saved_models/tft_nf"
    os.makedirs(save_path, exist_ok=True)
    nf_best.save(path=save_path, overwrite=True, save_dataset=False)
    mlflow.log_artifacts(save_path, artifact_path="tft_nf")
    print("Saved TFT model to", save_path)

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | padder_train            | ConstantPad1d            | 0      | train
2 | scaler                  | TemporalNorm             | 0      | train
3 | embedding               | TFTEmbedding             | 128    | train
4 | temporal_encoder        | TemporalCovariateEncoder | 39.6 K | train
5 | temporal_fusion_decoder | TemporalFusionDecoder    | 17.0 K | train
6 | output_adapter          | Linear                   | 33     | train
-----------------------------------------------------------------------------
56.8 K    Trainable params
0         Non-trainable params
56.8 K    Total params
0.227     Total estimated model params size (MB)
88        Modules in train mode
0         Modul

Epoch 0:  95%|█████████▌| 100/105 [02:04<00:06,  0.80it/s, v_num=10, train_loss_step=30.10]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 95/95 [02:03<00:00,  0.77it/s, v_num=10, train_loss_step=22.10, train_loss_epoch=67.50]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 95/95 [02:04<00:00,  0.76it/s, v_num=10, train_loss_step=22.10, train_loss_epoch=88.20]

`Trainer.fit` stopped: `max_steps=200` reached.


Epoch 1: 100%|██████████| 95/95 [02:04<00:00,  0.76it/s, v_num=10, train_loss_step=22.10, train_loss_epoch=88.20]
Saved TFT model to saved_models/tft_nf
🏃 View run TFT_Best_Pipeline at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/7/runs/032931768bd04644abf0c6c5f20bc172
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/7


## Notes

- WMAE here is on the real validation rows, comparable to the tree models.
- Params live in `configs/tft_config.yaml`; the model is built in
  `src/models/tft_pipeline.py`.
- TFT is the only model here that can use exogenous features, but they did not help.